In [ ]:
!pip -q install "transformers==4.46.*" "accelerate==1.1.*" "bitsandbytes==0.49.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 112.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import torch, subprocess

assert torch.cuda.is_available(), "no GPU: set Runtime > Change runtime type > T4 GPU"

gpu_name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print("gpu:", gpu_name)
print("total VRAM: %.2f GB" % total_gb)

print(subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip())

gpu: Tesla T4
total VRAM: 15.64 GB
3 MiB, 15360 MiB


In [ ]:
import torch, time, json, gc

def measured_vram_gb():
    torch.cuda.synchronize()
    return torch.cuda.memory_reserved(0) / 1e9

def free_vram():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(0)

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
print("helper ready for", MODEL_ID)

helper ready for Qwen/Qwen2.5-1.5B-Instruct


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)

before = measured_vram_gb()
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="cuda")
after = measured_vram_gb()

fp16_measured = after
print("fp16 resident VRAM: %.2f GB" % fp16_measured)
print("delta from before-load: %.2f GB" % (after - before))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

fp16 resident VRAM: 3.29 GB
delta from before-load: 3.29 GB


In [ ]:
from transformers import BitsAndBytesConfig

del model_fp16
free_vram()

model_int8 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    device_map="cuda")
int8_measured = measured_vram_gb()
print("int8 resident VRAM: %.2f GB" % int8_measured)

int8 resident VRAM: 1.87 GB


In [ ]:
del model_int8
free_vram()

model_int4 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True),
    device_map="cuda")
int4_measured = measured_vram_gb()
print("int4 resident VRAM: %.2f GB" % int4_measured)

int4 resident VRAM: 1.24 GB


In [ ]:
generate_src = '''
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

def load(dtype):
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    if dtype == "fp16":
        m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="cuda")
    elif dtype == "int8":
        m = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=BitsAndBytesConfig(load_in_8bit=True), device_map="cuda")
    elif dtype == "int4":
        m = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=BitsAndBytesConfig(load_in_4bit=True), device_map="cuda")
    else:
        raise ValueError(dtype)
    return tok, m

def tokens_per_s(dtype, new_tokens=128):
    tok, m = load(dtype)
    msgs = [{"role": "user", "content": "Explain what a GPU does, in three sentences."}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    m.generate(**{"input_ids": ids}, max_new_tokens=8)
    torch.cuda.synchronize()
    t0 = time.time()
    out = m.generate(**{"input_ids": ids}, max_new_tokens=new_tokens, do_sample=False)
    torch.cuda.synchronize()
    dt = time.time() - t0
    generated = out.shape[1] - ids.shape[1]
    return generated / dt

if __name__ == "__main__":
    for d in ["fp16", "int8", "int4"]:
        print(d, "%.1f tok/s" % tokens_per_s(d))
'''

with open("generate.py", "w") as f:
    f.write(generate_src)
print("wrote generate.py")

wrote generate.py


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("gen", "generate.py")
gen = importlib.util.module_from_spec(spec)

del model_int4
free_vram()
spec.loader.exec_module(gen)

fp16_tps = gen.tokens_per_s("fp16"); print("fp16 %.1f tok/s" % fp16_tps)
free_vram()
int8_tps = gen.tokens_per_s("int8"); print("int8 %.1f tok/s" % int8_tps)
free_vram()
int4_tps = gen.tokens_per_s("int4"); print("int4 %.1f tok/s" % int4_tps)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20`

fp16 28.5 tok/s
int8 5.6 tok/s


/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


int4 12.8 tok/s


In [ ]:
free_vram()
tok = AutoTokenizer.from_pretrained(MODEL_ID)
m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="cuda")
base = measured_vram_gb()
print("weights only: %.2f GB" % base)

for ctx in [256, 1024, 3072]:
    prompt = "word " * ctx
    ids = tok(prompt, return_tensors="pt").input_ids.to("cuda")
    m.generate(**{"input_ids": ids}, max_new_tokens=64, do_sample=False)
    peak = torch.cuda.max_memory_reserved(0) / 1e9
    print("ctx ~%d tokens -> peak VRAM %.2f GB (KV + activations: %.2f GB)" % (ctx, peak, peak - base))
    torch.cuda.reset_peak_memory_stats(0)

weights only: 3.31 GB
ctx ~256 tokens -> peak VRAM 3.34 GB (KV + activations: 0.03 GB)
ctx ~1024 tokens -> peak VRAM 3.44 GB (KV + activations: 0.13 GB)
ctx ~3072 tokens -> peak VRAM 3.65 GB (KV + activations: 0.34 GB)


In [ ]:
import json

results = {
    "model": MODEL_ID,
    "gpu": gpu_name,
    "measurements": [
        {"dtype": "fp16", "predicted_gb": None, "measured_gb": round(fp16_measured, 2), "tokens_per_s": round(fp16_tps, 1)},
        {"dtype": "int8", "predicted_gb": None, "measured_gb": round(int8_measured, 2), "tokens_per_s": round(int8_tps, 1)},
        {"dtype": "int4", "predicted_gb": None, "measured_gb": round(int4_measured, 2), "tokens_per_s": round(int4_tps, 1)},
    ],
}

results["measurements"][0]["predicted_gb"] = 3.0
results["measurements"][1]["predicted_gb"] = 1.5
results["measurements"][2]["predicted_gb"] = 0.75

with open("results.json", "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "gpu": "Tesla T4",
  "measurements": [
    {
      "dtype": "fp16",
      "predicted_gb": 3.0,
      "measured_gb": 3.29,
      "tokens_per_s": 28.5
    },
    {
      "dtype": "int8",
      "predicted_gb": 1.5,
      "measured_gb": 1.87,
      "tokens_per_s": 5.6
    },
    {
      "dtype": "int4",
      "predicted_gb": 0.75,
      "measured_gb": 1.24,
      "tokens_per_s": 12.8
    }
  ]
}


In [ ]:
import json, os


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def _fail(reason):
    print("GREEN CHECK: FAIL (%s)" % reason)
    raise _Stop()


def main():
    if not os.path.isfile("generate.py"):
        _fail("generate.py not found next to this cell")

    if not os.path.isfile("results.json"):
        _fail("results.json not found; run the results cell first")
    try:
        with open("results.json") as f:
            data = json.load(f)
    except json.JSONDecodeError as e:
        _fail("results.json is not valid JSON: %s" % e)

    for key in ("model", "gpu", "measurements"):
        if key not in data:
            _fail("results.json missing top-level key '%s'" % key)
    rows = data["measurements"]
    if not isinstance(rows, list) or len(rows) != 3:
        _fail("measurements must be a list of 3 rows (fp16, int8, int4)")

    by_dtype = {}
    for row in rows:
        for field in ("dtype", "predicted_gb", "measured_gb", "tokens_per_s"):
            if field not in row:
                _fail("a measurement row is missing '%s'" % field)
        dt = row["dtype"]
        if dt not in ("fp16", "int8", "int4"):
            _fail("unexpected dtype '%s'" % dt)
        for field in ("predicted_gb", "measured_gb", "tokens_per_s"):
            v = row[field]
            if not isinstance(v, (int, float)):
                _fail("%s.%s is not a number (fill it in)" % (dt, field))
            if v <= 0:
                _fail("%s.%s must be positive, got %s" % (dt, field, v))
        by_dtype[dt] = row

    for dt in ("fp16", "int8", "int4"):
        if dt not in by_dtype:
            _fail("missing the %s row" % dt)

    fp16 = by_dtype["fp16"]["measured_gb"]
    int8 = by_dtype["int8"]["measured_gb"]
    int4 = by_dtype["int4"]["measured_gb"]

    if not (2.5 <= fp16 <= 6.0):
        _fail("fp16 measured %.2f GB outside sane 2.5-6.0 GB; remeasure" % fp16)

    if not (int4 < int8 < fp16):
        _fail("memory order wrong: expected int4 < int8 < fp16, got %.2f, %.2f, %.2f"
              % (int4, int8, fp16))

    print("model:", data["model"])
    print("gpu:  ", data["gpu"])
    print("fp16 %.2f GB | int8 %.2f GB | int4 %.2f GB" % (fp16, int8, int4))
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    try:
        get_ipython()
    except NameError:
        raise SystemExit(1)

model: Qwen/Qwen2.5-1.5B-Instruct
gpu:   Tesla T4
fp16 3.29 GB | int8 1.87 GB | int4 1.24 GB
GREEN CHECK: PASS


In [ ]:
del m
free_vram()
m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32, device_map="cuda")
huge = "word " * 20000
ids = tok(huge, return_tensors="pt").input_ids.to("cuda")
m.generate(**{"input_ids": ids}, max_new_tokens=2000)

tensor([[1158, 3409, 3409,  ...,  353,  353,  353]], device='cuda:0')

In [ ]:
print(torch.cuda.memory_allocated(0) / 1e9, "GB allocated")
print(torch.cuda.memory_reserved(0) / 1e9, "GB reserved")
print(torch.cuda.max_memory_reserved(0) / 1e9, "GB peak reserved")

6.184776704 GB allocated
10.909384704 GB reserved
10.909384704 GB peak reserved


In [ ]:
CATALOG = {
    "Qwen2.5-0.5B-Instruct": {"hidden_size": 896, "num_hidden_layers": 24, "num_attention_heads": 14, "num_key_value_heads": 2, "intermediate_size": 4864, "vocab_size": 151936, "tie_word_embeddings": True},
    "Qwen2.5-1.5B-Instruct": {"hidden_size": 1536, "num_hidden_layers": 28, "num_attention_heads": 12, "num_key_value_heads": 2, "intermediate_size": 8960, "vocab_size": 151936, "tie_word_embeddings": True},
    "Qwen2.5-3B-Instruct": {"hidden_size": 2048, "num_hidden_layers": 36, "num_attention_heads": 16, "num_key_value_heads": 2, "intermediate_size": 11008, "vocab_size": 151936, "tie_word_embeddings": True},
    "Llama-3.2-1B-Instruct": {"hidden_size": 2048, "num_hidden_layers": 16, "num_attention_heads": 32, "num_key_value_heads": 8, "intermediate_size": 8192, "vocab_size": 128256, "tie_word_embeddings": True},
    "Llama-3.2-3B-Instruct": {"hidden_size": 3072, "num_hidden_layers": 28, "num_attention_heads": 24, "num_key_value_heads": 8, "intermediate_size": 8192, "vocab_size": 128256, "tie_word_embeddings": True},
}

In [ ]:
def count_params(cfg):
    h = cfg["hidden_size"]
    L = cfg["num_hidden_layers"]
    n_heads = cfg["num_attention_heads"]
    n_kv = cfg["num_key_value_heads"]
    inter = cfg["intermediate_size"]
    vocab = cfg["vocab_size"]
    tied = cfg.get("tie_word_embeddings", False)
    head_dim = h // n_heads

    q = h * h
    k = h * (n_kv * head_dim)
    v = h * (n_kv * head_dim)
    o = h * h
    attn = q + k + v + o

    gate = h * inter
    up = h * inter
    down = inter * h
    mlp = gate + up + down

    per_layer = attn + mlp
    embed_matrices = 1 if tied else 2
    total = L * per_layer + embed_matrices * vocab * h
    return total


for name, cfg in CATALOG.items():
    p = count_params(cfg)
    print(f"{name}: {p/1e9:.2f}B params")

Qwen2.5-0.5B-Instruct: 0.49B params
Qwen2.5-1.5B-Instruct: 1.54B params
Qwen2.5-3B-Instruct: 3.09B params
Llama-3.2-1B-Instruct: 1.24B params
Llama-3.2-3B-Instruct: 3.21B params


In [ ]:
cfg_modified = CATALOG["Qwen2.5-1.5B-Instruct"].copy()
cfg_modified["tie_word_embeddings"] = False

p_wrong = count_params(cfg_modified)
p_correct = count_params(CATALOG["Qwen2.5-1.5B-Instruct"])

print(f"correct (tied=True):  {p_correct/1e9:.2f}B")
print(f"wrong (tied=False):   {p_wrong/1e9:.2f}B")
print(f"overestimate: {(p_wrong/p_correct - 1)*100:.1f}%")

correct (tied=True):  1.54B
wrong (tied=False):   1.78B
overestimate: 15.1%


In [ ]:
def kv_bytes_per_token(cfg, kv_dtype_bytes=2):
    h = cfg["hidden_size"]
    L = cfg["num_hidden_layers"]
    n_kv = cfg["num_key_value_heads"]
    n_heads = cfg["num_attention_heads"]
    head_dim = h // n_heads
    return 2 * L * n_kv * head_dim * kv_dtype_bytes


for name, cfg in CATALOG.items():
    kb = kv_bytes_per_token(cfg)
    print(f"{name}: {kb} bytes/token")

Qwen2.5-0.5B-Instruct: 12288 bytes/token
Qwen2.5-1.5B-Instruct: 28672 bytes/token
Qwen2.5-3B-Instruct: 36864 bytes/token
Llama-3.2-1B-Instruct: 32768 bytes/token
Llama-3.2-3B-Instruct: 114688 bytes/token


In [ ]:
PRECISIONS = {"fp16": 2.0, "int8": 1.0, "int4": 0.5}
OVERHEAD_GB = 1.5

def solve(budget_gb, concurrent_users, context_tokens):
    rows = []
    for name, cfg in CATALOG.items():
        params = count_params(cfg)
        kv_per_tok = kv_bytes_per_token(cfg)
        for prec, bytes_per_param in PRECISIONS.items():
            weight_gb = params * bytes_per_param / 1e9
            kv_gb = kv_per_tok * context_tokens * concurrent_users / 1e9
            total_gb = weight_gb + kv_gb + OVERHEAD_GB
            fits = total_gb <= budget_gb
            rows.append({
                "model": name, "precision": prec, "params": params,
                "weight_gb": round(weight_gb, 2), "kv_gb": round(kv_gb, 2),
                "total_gb": round(total_gb, 2), "fits": fits,
            })
    fitting = [r for r in rows if r["fits"]]
    best = max(fitting, key=lambda r: r["params"]) if fitting else None
    return rows, best

rows, best = solve(budget_gb=16, concurrent_users=4, context_tokens=4096)
for r in sorted(rows, key=lambda r: -r["params"]):
    mark = "FITS" if r["fits"] else "----"
    print(f"{mark}  {r['model']:25s} {r['precision']:5s}  total={r['total_gb']:6.2f}GB")
print("BEST:", best)

FITS  Llama-3.2-3B-Instruct     fp16   total=  9.80GB
FITS  Llama-3.2-3B-Instruct     int8   total=  6.59GB
FITS  Llama-3.2-3B-Instruct     int4   total=  4.99GB
FITS  Qwen2.5-3B-Instruct       fp16   total=  8.28GB
FITS  Qwen2.5-3B-Instruct       int8   total=  5.19GB
FITS  Qwen2.5-3B-Instruct       int4   total=  3.65GB
FITS  Qwen2.5-1.5B-Instruct     fp16   total=  5.06GB
FITS  Qwen2.5-1.5B-Instruct     int8   total=  3.51GB
FITS  Qwen2.5-1.5B-Instruct     int4   total=  2.74GB
FITS  Llama-3.2-1B-Instruct     fp16   total=  4.51GB
FITS  Llama-3.2-1B-Instruct     int8   total=  3.27GB
FITS  Llama-3.2-1B-Instruct     int4   total=  2.65GB
FITS  Qwen2.5-0.5B-Instruct     fp16   total=  2.69GB
FITS  Qwen2.5-0.5B-Instruct     int8   total=  2.20GB
FITS  Qwen2.5-0.5B-Instruct     int4   total=  1.95GB
BEST: {'model': 'Llama-3.2-3B-Instruct', 'precision': 'fp16', 'params': 3212574720, 'weight_gb': 6.43, 'kv_gb': 1.88, 'total_gb': 9.8, 'fits': True}


In [ ]:
import json
budget_gb, concurrent_users, context_tokens = 16, 4, 4096
rows, best = solve(budget_gb, concurrent_users, context_tokens)
out = {
    "budget_gb": budget_gb, "concurrent_users": concurrent_users,
    "context_tokens": context_tokens, "best": best,
    "all_combinations": rows,
}
with open("budget_solution.json", "w") as f:
    json.dump(out, f, indent=2)
print(json.dumps(best, indent=2))

{
  "model": "Llama-3.2-3B-Instruct",
  "precision": "fp16",
  "params": 3212574720,
  "weight_gb": 6.43,
  "kv_gb": 1.88,
  "total_gb": 9.8,
  "fits": true
}


In [ ]:
import json, os
from typing import NoReturn

CATALOG = {
    "Qwen2.5-0.5B-Instruct": {"hidden_size": 896,  "num_hidden_layers": 24, "num_attention_heads": 14, "num_key_value_heads": 2, "intermediate_size": 4864,  "vocab_size": 151936, "tie_word_embeddings": True},
    "Qwen2.5-1.5B-Instruct": {"hidden_size": 1536, "num_hidden_layers": 28, "num_attention_heads": 12, "num_key_value_heads": 2, "intermediate_size": 8960,  "vocab_size": 151936, "tie_word_embeddings": True},
    "Qwen2.5-3B-Instruct":   {"hidden_size": 2048, "num_hidden_layers": 36, "num_attention_heads": 16, "num_key_value_heads": 2, "intermediate_size": 11008, "vocab_size": 151936, "tie_word_embeddings": True},
    "Llama-3.2-1B-Instruct": {"hidden_size": 2048, "num_hidden_layers": 16, "num_attention_heads": 32, "num_key_value_heads": 8, "intermediate_size": 8192,  "vocab_size": 128256, "tie_word_embeddings": True},
    "Llama-3.2-3B-Instruct": {"hidden_size": 3072, "num_hidden_layers": 28, "num_attention_heads": 24, "num_key_value_heads": 8, "intermediate_size": 8192,  "vocab_size": 128256, "tie_word_embeddings": True},
}
PRECISIONS = {"fp16": 2.0, "int8": 1.0, "int4": 0.5}
SCENARIO = {"budget_gb": 16, "concurrent_users": 4, "context_tokens": 4096}
OVERHEAD_GB = 1.5
TOL_GB = 0.06


class _Stop(Exception):
    pass


def _fail(reason) -> NoReturn:
    print("GREEN CHECK: FAIL (%s)" % reason)
    raise _Stop()


def count_params(cfg):
    h, L = cfg["hidden_size"], cfg["num_hidden_layers"]
    head_dim = h // cfg["num_attention_heads"]
    kv = cfg["num_key_value_heads"] * head_dim
    attn = h * h + h * kv + h * kv + h * h
    mlp = 3 * h * cfg["intermediate_size"]
    embeds = (1 if cfg.get("tie_word_embeddings") else 2) * cfg["vocab_size"] * h
    return L * (attn + mlp) + embeds


def kv_bytes_per_token(cfg, kv_dtype_bytes=2):
    head_dim = cfg["hidden_size"] // cfg["num_attention_heads"]
    return 2 * cfg["num_hidden_layers"] * cfg["num_key_value_heads"] * head_dim * kv_dtype_bytes


def truth():
    rows = {}
    for name, cfg in CATALOG.items():
        p = count_params(cfg)
        kv_gb = (kv_bytes_per_token(cfg) * SCENARIO["context_tokens"]
                 * SCENARIO["concurrent_users"] / 1e9)
        for prec, bpp in PRECISIONS.items():
            w = p * bpp / 1e9
            total = w + kv_gb + OVERHEAD_GB
            rows[(name, prec)] = {"params": p, "weight_gb": w, "kv_gb": kv_gb,
                                  "total_gb": total,
                                  "fits": total <= SCENARIO["budget_gb"]}
    return rows


def main():
    if not os.path.isfile("budget_solution.json"):
        _fail("budget_solution.json not found next to this script; run Step 5 first")
    try:
        with open("budget_solution.json") as f:
            sol = json.load(f)
    except json.JSONDecodeError as e:
        _fail("budget_solution.json is not valid JSON: %s" % e)

    for key in ("budget_gb", "concurrent_users", "context_tokens", "best", "all_combinations"):
        if key not in sol:
            _fail("missing top-level key '%s'" % key)
    for key, want in SCENARIO.items():
        if sol[key] != want:
            _fail("scenario %s=%r; the green check runs against the canonical "
                  "scenario %r (Step 5's numbers)" % (key, sol[key], want))

    rows = sol["all_combinations"]
    if not isinstance(rows, list) or len(rows) != len(CATALOG) * len(PRECISIONS):
        _fail("all_combinations must hold %d rows (5 models x 3 precisions), got %s"
              % (len(CATALOG) * len(PRECISIONS), len(rows) if isinstance(rows, list) else type(rows).__name__))

    t = truth()
    seen = set()
    for r in rows:
        key = (r.get("model"), r.get("precision"))
        if key not in t:
            _fail("unknown model/precision pair %r" % (key,))
        if key in seen:
            _fail("duplicate row for %r" % (key,))
        seen.add(key)
        want = t[key]
        for field in ("weight_gb", "kv_gb", "total_gb"):
            got = r.get(field)
            if not isinstance(got, (int, float)):
                _fail("%s missing numeric %s" % (key, field))
            if abs(got - want[field]) > TOL_GB:
                _fail("%s %s=%.2f, expected %.2f (formula or unit slip; "
                      "check tie_word_embeddings and GQA kv width first)"
                      % (key, field, got, want[field]))
        if bool(r.get("fits")) != want["fits"]:
            _fail("%s fits=%r, expected %r" % (key, r.get("fits"), want["fits"]))

    best = sol["best"]
    fitting = {k: v for k, v in t.items() if v["fits"]}
    if not fitting:
        _fail("verifier bug: canonical scenario should have fitting combos")
    want_key = max(fitting, key=lambda k: fitting[k]["params"])
    if best is None:
        _fail("best is null but %s/%s fits the budget" % want_key)
    if (best.get("model"), best.get("precision")) != want_key:
        _fail("best is %s/%s; the largest fitting combination is %s/%s"
              % (best.get("model"), best.get("precision"), *want_key))

    print("checked %d combinations against an independent recomputation" % len(rows))
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    try:
        get_ipython()
    except NameError:
        raise SystemExit(1)

checked 15 combinations against an independent recomputation
GREEN CHECK: PASS
